In [1]:
import os
import shutil
from typing import List, Tuple
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from tqdm import tqdm
from itertools import combinations

## Parameters

In [2]:
variables = []
with open("00 Variables.txt", "r") as f:
    lines = f.readlines()
    hashes = 1
    for line in lines:
        s = line.strip()
        if not s:
            continue
        if s.startswith("#"):
            hashes = len(s) - len(s.lstrip("#"))
            comment_indent = (hashes - 1) * 2
            print(" " * comment_indent + "\033[92m" + s + "\033[0m")
        elif "=" in s:
            var = s.split("=", 1)[0].strip()
            variables.append(var)
            try:
                exec(s)
                black_indent = {1: 2, 2: 5, 3: 9, 4: 14}.get(hashes, (hashes - 1) * 2 + sum(range(hashes + 1)))
                value_part = s.split("#", 1)[0].strip()
                comment_part = s[s.find("#"):] if "#" in s else ""
                eval_value = f"{var} = {repr(eval(var))}"
                if comment_part:
                    print(" " * black_indent + eval_value + " \033[38;5;250m" + "  " + comment_part + "\033[0m")
                else:
                    print(" " * black_indent + eval_value)
            except Exception as e:
                black_indent = {1: 2, 2: 5, 3: 9, 4: 14}.get(hashes, (hashes - 1) * 2 + sum(range(hashes + 1)))
                print(" " * black_indent + s)

# Global Settings
  TICKER = '^GSPC'
  START_DATE_GLOBAL = '1998-01-01'
  START_DATE = '2024-12-20'
  END_DATE = '2025-06-30'
  WINDOW_SIZE = 120
  HORIZON_DAYS = 1
  CRASH_PERCENTILE = 0.05
  MAX_WINDOWS = 10000
  DATE_INFO = True
  INDICATOR_SIZE = 'small'   # "small", "large"
  CHART_NUM = 4
  TRANSPARENT = False
  NEWS_QUERY = 'Standard and Poors 500 stock index'
  api_key = 'your-api-key'
  MODEL_NAME = 'gpt-4.1'   # "gpt-4.1", "o3-2025-04-16", "gpt-4.1-nano-2025-04-14"
  EMBEDDING_MODEL = 'text-embedding-3-small'
  RUN_TIMESTAMP = '20250726 1012'   # This will be changed automatically
  N_PREV = 6   # Number of previous windows to provide context for
  USE_IMAGES = True   # If True, send images (multimodal model), else just text
# Path
  ## Layer 1
     L1_RESULTS = 'Results'
     L1_COMB_RESULTS = 'Combination Results'
  ## Layer 2
     L2_DATA = 'Data'
     L2_FIGURES = 'Figures'
     L2_OUTPUT = 'Output'
  ## Layer 3
     L3_FIGURES_CHARTS = 'Generate Charts'
# Output Name
  O

## Definition

In [133]:
def clean_folder(folder_path: str):
    print(f"[INFO] Cleaning folder: {folder_path}")
    if os.path.exists(folder_path):
        deleted_any = False
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                    deleted_any = True
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
                    deleted_any = True
            except Exception as e:
                print(f"[ERROR] Failed to delete {file_path}. Reason: {e}\n")
        if deleted_any:
            print(f"[DELETE] Folder cleaned: {folder_path}\n")
        else:
            print(f"[INFO] Folder exists but is already clean: {folder_path}\n")
    else:
        os.makedirs(folder_path, exist_ok=True)
        print(f"[CREATE] Folder created: {folder_path}\n")

### Each Panel's Definition

In [134]:
def plot_panel_1(ax, window, x_vals, xticks, xticklabels):
    ax.plot(x_vals, window['close'].values, label="Close", color="black", lw=3)
    for name, color in [("sma20", "green"), ("sma50", "blue"), ("sma200", "orange")]:
        if name in window:
            ax.plot(x_vals, window[name].values, label=name.upper(), lw=4, linestyle="--", color=color)

    if 'bb_lower' in window and 'bb_upper' in window:
        ax.plot(x_vals, window['bb_lower'].values, label="BB Lower", color="red", lw=4, linestyle=":")
        ax.plot(x_vals, window['bb_upper'].values, label="BB Upper", color="green", lw=4, linestyle=":")
        ax.fill_between(x_vals, window['bb_upper'].values, window['bb_lower'].values, alpha=0.1, color="red", label="BB Band")

    if 'death_cross' in window:
        death_indices = [i for i, val in enumerate(window['death_cross']) if val == 1]
        if death_indices:
            ax.scatter([x_vals[i] for i in death_indices], window['close'].iloc[death_indices], color="red", marker="v", s=150, label="Death Cross", zorder=5)

    if 'golden_cross' in window:
        golden_indices = [i for i, val in enumerate(window['golden_cross']) if val == 1]
        if golden_indices:
            ax.scatter([x_vals[i] for i in golden_indices], window['close'].iloc[golden_indices],
                        color="gold", marker="^", s=150, label="Golden Cross", zorder=5)

    ax.set_ylabel("Price & Trend", fontsize=20, fontweight='bold')
    ax.yaxis.get_offset_text().set_fontsize(20)
    ax.yaxis.get_offset_text().set_fontweight('bold')
    ax.tick_params(axis='y', labelsize=20, width=1.5)
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    ax.set_xlabel("Date" if DATE_INFO == True else "Days in Window", fontsize=20, fontweight='bold')
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=0, ha='center')  # Centered on tick
    ax.tick_params(axis='x', labelsize=20, width=1.5, length=6, pad=12)
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')  # Make ticks bold
        label.set_rotation(0)         # Ensure no rotation

    # Add padding to x-axis limits
    if DATE_INFO == True:
        left_pad = pd.Timedelta(days=2)
        right_pad = pd.Timedelta(days=2)
        ax.set_xlim([x_vals[0] - left_pad, x_vals[-1] + right_pad])
    else:
        padding = int(len(x_vals) * 0.05) or 1
        ax.set_xlim([x_vals[0] - padding, x_vals[-1] + padding])

    ax.legend(loc="upper left", fontsize=18, prop={'weight': 'bold', 'size': 18}, markerscale=1, borderpad=0.75, framealpha=0.8, handlelength=1.5, handleheight=1)
    ax.get_legend().get_frame().set_edgecolor('black')
    ax.get_legend().get_frame().set_linewidth(2)
    ax.grid(True, linestyle='--', alpha=0.3)


def plot_panel_2(ax, window, x_vals, xticks, xticklabels):
    ax.set_ylabel("Volatility", color="black", fontsize=20, fontweight='bold')
    ax.yaxis.get_offset_text().set_fontsize(20)
    ax.yaxis.get_offset_text().set_fontweight('bold')
    ax.tick_params(axis='y', labelsize=20, width=1.5)
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')

    for name, color in [
        ("volatility", "brown"), ("hv_10", "green"), ("hv_30", "orange"),
        ("vol_parkinson", "purple"), ("vol_gk", "cyan"), ("vol_rs", "magenta"),
        ("composite_vol", "grey"), ("volofvol", "black")
    ]:
        if name in window:
            ax.plot(x_vals, window[name].values, label=name.upper(), lw=3, color=color)
    
    for q, color in zip([1, 5, 95, 99], ['red', 'orange', 'green', 'blue']):
        colname = f"volatility_{q}pct"
        if colname in window:
            ax.plot(x_vals, window[colname].values, label=f"Vol_{q}pct", color=color, lw=4, linestyle=":")

    ax.legend(loc="upper left", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax.get_legend().get_frame().set_edgecolor('black')
    ax.get_legend().get_frame().set_linewidth(2)

    ax.set_xlabel("Date" if DATE_INFO == True else "Days in Window", fontsize=20, fontweight='bold')
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=0, ha='center')
    ax.tick_params(axis='x', labelsize=20, width=1.5, length=6, pad=12)
    
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')  # make ticks bold

    # Add padding to x-axis limits
    if DATE_INFO == True:
        left_pad = pd.Timedelta(days=2)
        right_pad = pd.Timedelta(days=2)
        ax.set_xlim([x_vals[0] - left_pad, x_vals[-1] + right_pad])
    else:
        padding = int(len(x_vals) * 0.05) or 1
        ax.set_xlim([x_vals[0] - padding, x_vals[-1] + padding])

    ax.grid(True, linestyle='--', alpha=0.3)


def plot_panel_3(ax, window, x_vals, xticks, xticklabels):
    if 'macd' in window:
        ax.plot(x_vals, window['macd'].values, label="MACD", lw=3, color="blue")

    if 'macd_signal' in window:
        ax.plot(x_vals, window['macd_signal'].values, label="MACD Signal", color="orange", linestyle="--", lw=4)
    
    if 'macd_hist' in window:
        colors = ['green' if x >= 0 else 'red' for x in window['macd_hist']]
        ax.bar(x_vals, window['macd_hist'].values, alpha=0.3, label="MACD Hist", color=colors)
    
    if 'rsi' in window:
        ax.plot(x_vals, window['rsi'].values, label="RSI", color="purple", lw=3)
        ax.plot(x_vals, [70] * len(x_vals), color='red', linestyle='--', lw=3, alpha=0.5, label="Overbought (70)")
        ax.plot(x_vals, [30] * len(x_vals), color='green', linestyle='--', lw=3, alpha=0.5, label="Oversold (30)")

    ax.axhline(0, color='black', linestyle='-', lw=1.5, alpha=0.2)
    ax.set_ylabel("Oscillators",color="black", fontsize=20, fontweight='bold')
    ax.yaxis.get_offset_text().set_fontsize(20)
    ax.yaxis.get_offset_text().set_fontweight('bold')

    ax.legend(loc="upper left", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax.get_legend().get_frame().set_edgecolor('black')
    ax.get_legend().get_frame().set_linewidth(2)

    ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.3)
    ax.tick_params(axis='y', labelsize=20, width=1.5)
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    # X-axis formatting
    ax.set_xlabel("Date" if DATE_INFO == True else "Days in Window", fontsize=20, fontweight='bold')
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=0, ha='center')
    ax.tick_params(axis='x', labelsize=20, width=2, length=6, pad=12)
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')  # make ticks bold
    
    # Add padding to x-axis
    if DATE_INFO == True:
        left_pad = pd.Timedelta(days=2)
        right_pad = pd.Timedelta(days=2)
        ax.set_xlim([x_vals[0] - left_pad, x_vals[-1] + right_pad])
    else:
        padding = int(len(x_vals) * 0.05) or 1
        ax.set_xlim([x_vals[0] - padding, x_vals[-1] + padding])


def plot_panel_4(ax, window, x_vals, xticks, xticklabels):
    # Calculate padding for x-axis
    if DATE_INFO == True:
        left_pad = x_vals[0] - pd.Timedelta(days=2)
        right_pad = x_vals[-1] + pd.Timedelta(days=2)
    else:
        padding = int(len(x_vals) * 0.05) or 1
        left_pad = x_vals[0] - padding
        right_pad = x_vals[-1] + padding

    # Helper to extend series with padding NaNs for smooth line start/end
    def extend_series(x_vals, y_vals):
        from numpy import nan
        new_x = [left_pad] + list(x_vals) + [right_pad]
        new_y = [nan] + list(y_vals) + [nan]
        return new_x, new_y

    # Plot ROC lines
    for name, color in [("roc_5", "blue"), ("roc_10", "orange"), ("roc_20", "green")]:
        if name in window:
            new_x, new_y = extend_series(x_vals, window[name].values)
            ax.plot(new_x, new_y, label=name.upper(), lw=3, color=color)

    # Plot Return quantile lines
    for q, color in zip([1, 5, 95, 99], ['red', 'orange', 'green', 'blue']):
        colname = f"return_{q}pct_pct"
        if colname in window:
            new_x, new_y = extend_series(x_vals, window[colname].values)
            ax.plot(new_x, new_y, label=f"Ret_{q}pct", color=color, lw=4, linestyle=":")

    # Draw zero line with same padding
    ax.plot([left_pad, right_pad], [0, 0], color='black', linestyle='-', lw=2, alpha=0.3)

    ax.set_ylabel("Momentum/ROC (%)",color="black", fontsize=20, fontweight='bold')
    ax.yaxis.get_offset_text().set_fontsize(20)
    ax.yaxis.get_offset_text().set_fontweight('bold')
    ax.tick_params(axis='y', labelsize=20, width=1.5)
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    ax.legend(loc="upper left", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax.get_legend().get_frame().set_edgecolor('black')
    ax.get_legend().get_frame().set_linewidth(2)

    ax.grid(True, alpha=0.3)

    # X-axis formatting
    ax.set_xlabel("Date" if DATE_INFO == True else "Days in Window", fontsize=20, fontweight='bold')
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=0, ha='center')
    ax.tick_params(axis='x', labelsize=20, width=1.5, length=6, pad=12)
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')

    ax.set_xlim([left_pad, right_pad])

def plot_panel_5(ax, window, x_vals, xticks, xticklabels):
    # Plot ATR
    if 'atr' in window:
        ax.plot(x_vals, window['atr'].values, label="ATR", color="red", lw=3)

    # Create a twin axis for volume bars
    ax2 = ax.twinx()
    if 'volume' in window:
        ax2.bar(x_vals, window['volume'].values, label="Volume", color='grey', alpha=0.4)

    # Set y-labels
    ax.set_ylabel("ATR",color="black", fontsize=20, fontweight='bold')
    ax.yaxis.get_offset_text().set_fontsize(20)
    ax.yaxis.get_offset_text().set_fontweight('bold')
    ax2.set_ylabel("Volume",color="black", fontsize=20, fontweight='bold')
    ax2.yaxis.get_offset_text().set_fontsize(20)
    ax2.yaxis.get_offset_text().set_fontweight('bold')

    ax.tick_params(axis='y', labelsize=20, width=1.5)
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    ax2.tick_params(axis='y', labelsize=20, width=1.5)
    for label in ax2.get_yticklabels():
        label.set_fontweight('bold')
        
    # Legends on both axes
    ax.legend(loc="upper left", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax.get_legend().get_frame().set_edgecolor('black')
    ax.get_legend().get_frame().set_linewidth(2)

    ax2.legend(loc="upper right", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax2.get_legend().get_frame().set_edgecolor('black')
    ax2.get_legend().get_frame().set_linewidth(2)

    # Grid for primary axis
    ax.grid(True, alpha=0.3)

    # X-axis formatting
    ax.set_xlabel("Date" if DATE_INFO == True else "Days in Window", fontsize=20, fontweight='bold')
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=0, ha='center')
    ax.tick_params(axis='x', labelsize=20, width=1.5, length=6, pad=12)
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')  # make labels bold

    # Add x-axis padding
    if DATE_INFO == True:
        ax.set_xlim([x_vals[0] - pd.Timedelta(days=2), x_vals[-1] + pd.Timedelta(days=2)])
    else:
        padding = int(len(x_vals) * 0.05) or 1
        ax.set_xlim([x_vals[0] - padding, x_vals[-1] + padding])

def plot_panel_6(ax, window, x_vals, xticks, xticklabels):
    if 'cmf' in window:
        ax.plot(x_vals, window['cmf'].values, label="CMF", color="blue", lw=3)
        ax.axhline(0, color='black', linestyle='-', lw=2, alpha=0.3)

    ax.set_ylabel("CMF",color="black", fontsize=20, fontweight='bold')
    ax.yaxis.get_offset_text().set_fontsize(20)
    ax.yaxis.get_offset_text().set_fontweight('bold')
    ax.tick_params(axis='y', labelsize=20, width=1.5)
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    ax.legend(loc="upper left", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax.get_legend().get_frame().set_edgecolor('black')
    ax.get_legend().get_frame().set_linewidth(2)

    ax.grid(True, alpha=0.3)

    # X-Axis formatting
    ax.set_xlabel("Date" if DATE_INFO == True else "Days in Window", fontsize=20, fontweight='bold')
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=0, ha='center')
    ax.tick_params(axis='x', labelsize=20, width=1.5, length=6, pad=12)
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')  # make thicker (bold)

    # Add padding to x-axis
    if DATE_INFO == True:
        ax.set_xlim([x_vals[0] - pd.Timedelta(days=2), x_vals[-1] + pd.Timedelta(days=2)])
    else:
        padding = int(len(x_vals) * 0.05) or 1
        ax.set_xlim([x_vals[0] - padding, x_vals[-1] + padding])


def plot_panel_7(ax, window, x_vals, xticks, xticklabels):
    # Prepare x axis start and end
    x_start = x_vals[0]
    x_end = x_vals[-1]
    x_plot = x_vals

    # Plot Stoch and Williams %R on primary y-axis
    if 'stoch_k' in window:
        ax.plot(x_plot, window['stoch_k'].values, label="Stoch %K", color="blue", lw=3)
    if 'stoch_d' in window:
        ax.plot(x_plot, window['stoch_d'].values, label="Stoch %D", color="cyan", lw=3)
    if 'williamsr' in window:
        ax.plot(x_plot, window['williamsr'].values, label="Williams %R", color="purple", lw=3)

    # Horizontal threshold lines on primary axis
    ax.plot([x_start, x_end], [80, 80], color='blue', linestyle=':', alpha=0.5, lw=4, label='Upper Bound (80)')
    ax.plot([x_start, x_end], [20, 20], color='blue', linestyle=':', alpha=0.5, lw=4, label='Lower Bound (20)')
    ax.plot([x_start, x_end], [0, 0], color='black', linestyle='-', alpha=0.3, lw=2)

    # Secondary y-axis for Z-Score
    ax2 = ax.twinx()
    if 'zscore' in window:
        ax2.plot(x_plot, window['zscore'].values, label="Z-Score", color="green", lw=4, linestyle="--")
        ax2.plot([x_start, x_end], [2, 2], color='green', linestyle='--', alpha=0.7, lw=4, label="+2 Threshold")
        ax2.plot([x_start, x_end], [-2, -2], color='green', linestyle='--', alpha=0.7, lw=4, label="-2 Threshold")

    # Labels
    ax.set_ylabel("Stoch / WilliamsR",color="black", fontsize=20, fontweight='bold')
    ax.yaxis.get_offset_text().set_fontsize(20)
    ax.yaxis.get_offset_text().set_fontweight('bold')
    ax2.set_ylabel("Z-Score",color="black", fontsize=20, fontweight='bold')
    ax2.yaxis.get_offset_text().set_fontsize(20)
    ax2.yaxis.get_offset_text().set_fontweight('bold')
    ax.tick_params(axis='y', labelsize=20, width=1.5)
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    ax2.tick_params(axis='y', labelsize=20, width=1.5)
    for label in ax2.get_yticklabels():
        label.set_fontweight('bold')
        
    # Legends from both axes
    lines_1, labels_1 = ax.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax.legend(lines_1, labels_1, loc="upper left", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax.get_legend().get_frame().set_edgecolor('black')
    ax.get_legend().get_frame().set_linewidth(2)

    ax2.legend(lines_2, labels_2, loc="upper right", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax2.get_legend().get_frame().set_edgecolor('black')
    ax2.get_legend().get_frame().set_linewidth(2)

    ax.grid(True, alpha=0.3)

    # X-Axis formatting
    ax.set_xlabel("Date" if DATE_INFO == True else "Days in Window", fontsize=20, fontweight='bold')
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=0, ha='center')
    ax.tick_params(axis='x', labelsize=20, width=1.5, length=6, pad=12)
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')  # bold ticks

    # Axis limits with padding
    if DATE_INFO == True:
        ax.set_xlim([x_start - pd.Timedelta(days=2), x_end + pd.Timedelta(days=2)])
    else:
        padding = int(len(x_vals) * 0.05) or 1
        ax.set_xlim([x_start - padding, x_end + padding])

def plot_panel_8(ax, window, x_vals, xticks, xticklabels):
    ax2 = ax.twinx()
    lines2 = []
    labels2 = []

    # Determine padded x-axis limits
    if DATE_INFO == True:
        left_pad = x_vals[0] - pd.Timedelta(days=2)
        right_pad = x_vals[-1] + pd.Timedelta(days=2)
        left_pad_val = left_pad
        right_pad_val = right_pad
        x_vals_plot = x_vals
    else:
        padding = int(len(x_vals) * 0.05) or 1
        left_pad_val = x_vals[0] - padding
        right_pad_val = x_vals[-1] + padding
        x_vals_plot = x_vals

    # Threshold lines use exact data start/end (no padding)
    threshold_start = x_vals[0]
    threshold_end = x_vals[-1]

    # Plot CCI with thresholds on secondary y-axis
    if 'cci' in window:
        line_cci, = ax2.plot(x_vals_plot, window['cci'].values, label="CCI", color="blue", lw=3)
        ax2.plot([threshold_start, threshold_end], [100, 100], color='red', linestyle='--', lw=4, alpha=0.5)
        ax2.plot([threshold_start, threshold_end], [-100, -100], color='green', linestyle='--', lw=4, alpha=0.5)
        line_100 = Line2D([], [], color='red', linestyle='--', alpha=0.5, label="+100 Threshold")
        line_m100 = Line2D([], [], color='green', linestyle='--', alpha=0.5, label="-100 Threshold")
        lines2.extend([line_cci, line_100, line_m100])
        labels2.extend(["CCI", "+100 Threshold", "-100 Threshold"])
        ax2.set_ylabel("CCI", color="black", fontsize=20, fontweight='bold')
        ax2.yaxis.get_offset_text().set_fontsize(20)
        ax2.yaxis.get_offset_text().set_fontweight('bold')
        ax2.tick_params(axis='y', labelsize=20, width=1.5)
        for label in ax2.get_yticklabels():
            label.set_fontweight('bold')

    # Plot other indicators on primary y-axis
    if 'ulcer_index' in window:
        ax.plot(x_vals_plot, window['ulcer_index'].values, label="Ulcer Index", lw=3, color="red")
    if 'tr_percent' in window:
        ax.plot(x_vals_plot, window['tr_percent'].values, label="TR %", lw=3, color="orange")
    if 'sharpe_20' in window:
        ax.plot(x_vals_plot, window['sharpe_20'].values, label="Sharpe 20d", lw=3, color="green", linestyle='--')
    if 'maxdd_20' in window:
        ax.plot(x_vals_plot, window['maxdd_20'].values, label="MaxDD 20d", lw=3, color="black", linestyle='--')

    # Zero line on primary axis (for aesthetics)
    ax.plot([left_pad_val, right_pad_val], [0, 0], color='black', linestyle='-', lw=4, alpha=0.3)

    ax.set_ylabel("Ulcer / TR% / Sharpe / MaxDD", color="black", fontsize=20, fontweight='bold')
    ax.yaxis.get_offset_text().set_fontsize(20)
    ax.yaxis.get_offset_text().set_fontweight('bold')
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    ax.tick_params(axis='y', labelsize=20, width=1.5)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Date" if DATE_INFO == True else "Days in Window", fontsize=20, fontweight='bold')

    # Set x-axis limits with padding
    ax.set_xlim([left_pad_val, right_pad_val])

    # Set x-ticks and labels properly
    if DATE_INFO == True:
        ax.set_xticks(xticks)
        ax.set_xticklabels(xticklabels, rotation=0, ha='center')
    else:
        ax.set_xticks(list(range(0, len(window), max(1, len(window)//10))))

    ax.tick_params(axis='x', labelsize=20, width=1.5, length=6, pad=12)
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')

    # Legends for both axes
    lines1, labels1 = ax.get_legend_handles_labels()
    ax.legend(lines1, labels1, loc="upper left", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax.get_legend().get_frame().set_edgecolor('black')
    ax.get_legend().get_frame().set_linewidth(2)

    ax2.legend(lines2, labels2, loc="upper right", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax2.get_legend().get_frame().set_edgecolor('black')
    ax2.get_legend().get_frame().set_linewidth(2)


def plot_panel_9(ax, window, x_vals, xticks, xticklabels):
    ax2 = ax.twinx()

    if 'obv' in window:
        ax.plot(x_vals, window['obv'].values, label="OBV", lw=3, color="blue")
    ax.set_ylabel("OBV", color="black", fontsize=20, fontweight='bold')
    ax.yaxis.get_offset_text().set_fontsize(20)
    ax.yaxis.get_offset_text().set_fontweight('bold')
    ax.tick_params(axis='y', labelsize=20, width=1.5)

    if 'adx' in window:
        ax2.plot(x_vals, window['adx'].values, label="ADX", color="red", lw=3)

        threshold_start = x_vals[0]
        threshold_end = x_vals[-1]
        ax2.plot([threshold_start, threshold_end], [25, 25],
                 color='orange', linestyle='--', lw=4, alpha=0.5, label='Strong Trend (ADX > 25)')

        ax2.set_ylabel("ADX", color="black", fontsize=20, fontweight='bold')
        ax2.yaxis.get_offset_text().set_fontsize(20)
        ax2.yaxis.get_offset_text().set_fontweight('bold')

    # Set tick params for both y-axes
    ax.tick_params(axis='y', labelsize=20, width=1.5)
    ax2.tick_params(axis='y', labelcolor="black", labelsize=20, width=1.5)

    # Now make y-tick labels bold for both y-axes
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    for label in ax2.get_yticklabels():
        label.set_fontweight('bold')

    ax.grid(True, alpha=0.3)

    ax.set_xlabel("Date" if DATE_INFO == True else "Days in Window",
                  fontsize=20, fontweight='bold')
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=0, ha='center')
    ax.tick_params(axis='x', labelsize=20, width=1.5, length=6, pad=12)
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')

    if DATE_INFO == True:
        left_pad = pd.Timedelta(days=2)
        right_pad = pd.Timedelta(days=2)
        ax.set_xlim([x_vals[0] - left_pad, x_vals[-1] + right_pad])
    else:
        padding = int(len(x_vals) * 0.05) or 1
        ax.set_xlim([x_vals[0] - padding, x_vals[-1] + padding])

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1, labels1, loc="upper left", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax.get_legend().get_frame().set_edgecolor('black')
    ax.get_legend().get_frame().set_linewidth(2)
    
    ax2.legend(lines2, labels2, loc="upper right", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax2.get_legend().get_frame().set_edgecolor('black')
    ax2.get_legend().get_frame().set_linewidth(2)


def plot_panel_10(ax, window, x_vals, xticks, xticklabels, include_distributions=True, n_panels=10):
    if not (include_distributions and n_panels >= 10):
        # If panel is not included, just return or plot minimal placeholder
        ax.set_title("Skewness & Kurtosis (not available)")
        return

    # Determine padded axis limits for x-axis
    if DATE_INFO == True:
        left_pad = x_vals[0] - pd.Timedelta(days=2)
        right_pad = x_vals[-1] + pd.Timedelta(days=2)
        x_plot = x_vals
        zero_line_start = x_vals[0]
        zero_line_end = x_vals[-1]
    else:
        padding = int(len(x_vals) * 0.05) or 1
        left_pad = x_vals[0] - padding
        right_pad = x_vals[-1] + padding
        x_plot = x_vals
        zero_line_start = x_vals[0]
        zero_line_end = x_vals[-1]

    lines1, labels1 = [], []
    if 'skewness_20' in window:
        line = ax.plot(x_plot, window['skewness_20'].values, label="Skewness (20d)", color="blue", lw=3)[0]
        lines1.append(line)
        labels1.append("Skewness (20d)")
    if 'skewness_60' in window:
        line = ax.plot(x_plot, window['skewness_60'].values, label="Skewness (60d)", color="cyan", lw=3)[0]
        lines1.append(line)
        labels1.append("Skewness (60d)")

    zero_skew_line, = ax.plot([zero_line_start, zero_line_end], [0, 0], 
                             color='skyblue', linestyle='--', lw=4, alpha=0.5, label="Skewness = 0")
    lines1.append(zero_skew_line)
    labels1.append("Skewness = 0")

    ax2 = ax.twinx()
    lines2, labels2 = [], []
    if 'kurtosis_20' in window:
        line = ax2.plot(x_plot, window['kurtosis_20'].values, label="Kurtosis (20d)", color="red", lw=3)[0]
        lines2.append(line)
        labels2.append("Kurtosis (20d)")
    if 'kurtosis_60' in window:
        line = ax2.plot(x_plot, window['kurtosis_60'].values, label="Kurtosis (60d)", color="orange", lw=3)[0]
        lines2.append(line)
        labels2.append("Kurtosis (60d)")

    zero_kurt_line, = ax2.plot([zero_line_start, zero_line_end], [0, 0],
                              color='lightcoral', linestyle='--', lw=4, alpha=0.5, label="Kurtosis = 0")
    lines2.append(zero_kurt_line)
    labels2.append("Kurtosis = 0")

    ax.set_ylabel("Skewness", color="black", fontsize=20, fontweight='bold')
    ax.yaxis.get_offset_text().set_fontsize(20)
    ax.yaxis.get_offset_text().set_fontweight('bold')
    ax2.set_ylabel("Kurtosis", color="black", fontsize=20, fontweight='bold')
    ax2.yaxis.get_offset_text().set_fontsize(20)
    ax2.yaxis.get_offset_text().set_fontweight('bold')
    ax.tick_params(axis='y', labelsize=20, width=1.5)
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
    ax2.tick_params(axis='y', labelsize=20, width=1.5)
    for label in ax2.get_yticklabels():
        label.set_fontweight('bold')

    ax.legend(lines1, labels1, loc="upper left", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax.get_legend().get_frame().set_edgecolor('black')
    ax.get_legend().get_frame().set_linewidth(2)

    ax2.legend(lines2, labels2, loc="upper right", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax2.get_legend().get_frame().set_edgecolor('black')
    ax2.get_legend().get_frame().set_linewidth(2)

    ax.grid(True, alpha=0.3)

    ax.set_xlabel("Date" if DATE_INFO == True else "Days in Window", fontsize=20, fontweight='bold')
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=0, ha='center')
    ax.tick_params(axis='x', labelsize=20, width=1.5, length=6, pad=12)
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')

    ax.set_xlim([left_pad, right_pad])

def plot_panel_11(ax, window, x_vals, xticks, xticklabels, include_distributions=True, n_panels=11):


    if not (include_distributions and n_panels >= 11):
        ax.set_title("VaR & ES (not available)")
        return

    if 'var_5pct_20' in window:
        ax.plot(x_vals, window['var_5pct_20'].values, label="5% VaR", color="red", lw=3)
    if 'var_1pct_20' in window:
        ax.plot(x_vals, window['var_1pct_20'].values, label="1% VaR", color="darkred", lw=3)
    if 'es_5pct_20' in window:
        ax.plot(x_vals, window['es_5pct_20'].values, label="5% ES", color="orange", lw=3)
    if 'logreturn' in window:
        ax.scatter(x_vals, window['logreturn'].values, alpha=0.3, s=10, color='gray', label='Daily Returns')

    # Zero line spanning exact data range (no padding)
    ax.plot([x_vals[0], x_vals[-1]], [0, 0], color='black', linestyle='-', lw=2, alpha=0.3)

    ax.set_xlabel("Date" if DATE_INFO == True else "Days in Window", fontsize=20, fontweight='bold')
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, rotation=0, ha='center')
    ax.tick_params(axis='x', labelsize=20, width=1.5, length=6, pad=12)
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')

    # Set x-axis limits with padding
    if DATE_INFO == True:
        left_pad = x_vals[0] - pd.Timedelta(days=2)
        right_pad = x_vals[-1] + pd.Timedelta(days=2)
        ax.set_xlim([left_pad, right_pad])
    else:
        padding = int(len(x_vals) * 0.05) or 1
        ax.set_xlim([x_vals[0] - padding, x_vals[-1] + padding])

    ax.set_ylabel("VaR & ES", color="black", fontsize=20, fontweight='bold')
    ax.yaxis.get_offset_text().set_fontsize(20)
    ax.yaxis.get_offset_text().set_fontweight('bold')
    ax.tick_params(axis='y', labelsize=20, width=1.5)
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')

    ax.legend(loc="upper left", fontsize=18, prop={'weight': 'bold', 'size': 18}, framealpha=0.8)
    ax.get_legend().get_frame().set_edgecolor('black')
    ax.get_legend().get_frame().set_linewidth(2)

    ax.grid(True, alpha=0.3)

In [135]:
def create_chart(window, ticker, date_str, panels, chart_index, save_fig_dir, _transparent=TRANSPARENT):
    fig, axs = plt.subplots(len(panels), 1, figsize=(40, 7 * len(panels)), sharex=False)
    axs = axs if isinstance(axs, np.ndarray) else [axs]

    if DATE_INFO == True:
        x_vals = window.index
        xticks = window.index[::max(1, len(window) // 10)]
        xticklabels = [d.strftime("%Y-%m-%d") for d in xticks]
    else:
        x_vals = list(range(len(window)))
        xticks = list(range(0, len(window), max(1, len(window)//10)))
        xticklabels = [str(i) for i in xticks]

    for i, panel_func in enumerate(panels):
        panel_func(axs[i], window, x_vals, xticks, xticklabels)

    filename = f"{ticker}_{date_str}_chart{chart_index}.png"

    chart_folder = os.path.join(save_fig_dir, f"Chart {chart_index}")
    os.makedirs(chart_folder, exist_ok=True)
    filepath = os.path.join(chart_folder, filename)
    
    title_str = f"{ticker} - {date_str.replace('-', '')}" if DATE_INFO else f"{ticker}"
    fig.suptitle(title_str, fontsize=28, fontweight='bold', y=0.995)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(filepath, transparent=_transparent, dpi=144)
    plt.close(fig)
    return filepath

def create_technical_chart(window, ticker, date_str, save_fig_dir="figures", include_distributions=True, chart_num=CHART_NUM):
    PANEL_REQUIREMENTS = {
        plot_panel_1:  ["close", "sma20", "sma50", "sma200", "bb_lower", "bb_upper", "death_cross", "golden_cross"],
        plot_panel_2:  ["volatility", "hv_10", "hv_30", "vol_parkinson", "vol_gk", "vol_rs", "composite_vol", "volofvol"],
        plot_panel_3:  ["macd", "macd_signal", "macd_hist", "rsi"],
        plot_panel_4:  ["roc_5", "roc_10", "roc_20", "return_1pct_pct", "return_5pct_pct", "return_95pct_pct", "return_99pct_pct"],
        plot_panel_5:  ["atr", "volume"],
        plot_panel_6:  ["cmf"],
        plot_panel_7:  ["stoch_k", "stoch_d", "williamsr", "zscore"],
        plot_panel_8:  ["cci", "ulcer_index", "tr_percent", "sharpe_20", "maxdd_20"],
        plot_panel_9:  ["obv", "adx"],
        plot_panel_10: ["skewness_20", "skewness_60", "kurtosis_20", "kurtosis_60"],
        plot_panel_11: ["var_5pct_20", "var_1pct_20", "es_5pct_20", "logreturn"],
    }

    panel_funcs_all = [
        plot_panel_1, plot_panel_2, plot_panel_3, plot_panel_4,
        plot_panel_5, plot_panel_6, plot_panel_7, plot_panel_8,
        plot_panel_9
    ]

    if include_distributions:
        panel_funcs_all += [plot_panel_10, plot_panel_11]

    available_cols = set(window.columns)
    panel_funcs = [
        pf for pf in panel_funcs_all
        if any(col in available_cols for col in PANEL_REQUIREMENTS[pf])
    ]
    if not panel_funcs:
        return tuple()

    n_panels = len(panel_funcs)
    panels_per_chart = [n_panels // chart_num] * chart_num
    for i in range(n_panels % chart_num):
        panels_per_chart[i] += 1

    charts = []
    start = 0
    for i in range(chart_num):
        end = start + panels_per_chart[i]
        if start >= n_panels:
            break
        chart = create_chart(window, ticker, date_str, panel_funcs[start:end], chart_index=i + 1, save_fig_dir=save_fig_dir)
        charts.append(chart)
        start = end

    return tuple(charts)

In [136]:
def generate_rolling_window_charts_only(df_ind, window_size, save_fig_dir, max_windows=MAX_WINDOWS, ticker=TICKER):
    first_valid_index = df_ind.first_valid_index()
    first_valid_pos = df_ind.index.get_loc(first_valid_index)
    start_pos = first_valid_pos + window_size
    end_pos = len(df_ind) + 1
    N = min(len(df_ind), start_pos + max_windows)

    with tqdm(total=N - start_pos, desc="Generating Rolling Charts") as pbar:
        for i in range(start_pos, N):
            window = df_ind.iloc[i - window_size: i]
            date_str = window.index[-1].strftime("%Y-%m-%d")
            try:
                image_paths = create_technical_chart(window, ticker, date_str, save_fig_dir=save_fig_dir)
                pbar.set_postfix_str(f"Saved {ticker}_{date_str} Series Plot")
            except Exception as e:
                print(f"[ERROR] Chart failed for {date_str}: {e}")
            pbar.update(1)

## Main Execution

In [137]:
FIG_DIR = f"{L1_RESULTS}/{L2_FIGURES}/{L3_FIGURES_CHARTS}"
clean_folder(FIG_DIR)

print(f"Inputing price data for {TICKER} from {START_DATE} to {END_DATE} ...")
df_prices = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{OHLCV}.csv", index_col=0, parse_dates=True)
print("Data inputted. Shape:", df_prices.shape, "\n")

print("Inputing indicators...")
df_indicators = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{INDICATORS}.csv", index_col=0, parse_dates=True)
print("Indicators inputed. Shape:", df_indicators.shape, "\n")

print("Generating rolling window charts (starting from first day with full indicator data)...")

[INFO] Cleaning folder: Results/Figures/Generate Charts
[DELETE] Folder cleaned: Results/Figures/Generate Charts

Inputing price data for ^GSPC from 2024-12-20 to 2025-06-30 ...
Data inputted. Shape: (6913, 6) 

Inputing indicators...
Indicators inputed. Shape: (6674, 69) 

Generating rolling window charts (starting from first day with full indicator data)...


In [138]:
print(f"Inputing price data for {TICKER} from {START_DATE} to {END_DATE} ...")
df = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{OHLCV}.csv", index_col=0, parse_dates=True)
df = df[(df.index >= START_DATE) & (df.index <= END_DATE)]
print("Data inputted. Shape:", df.shape, "\n")

print("Inputing indicators...")
if INDICATOR_SIZE == "small":
    INDICATORS_NAME = INDICATORS + "_" + INDICATOR_SIZE
else:
    INDICATORS_NAME = INDICATORS

df_indicators = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{INDICATORS_NAME}.csv", index_col=0, parse_dates=True)
df_indicators = df_indicators[(df_indicators.index >= START_DATE) & (df_indicators.index <= END_DATE)]
print("Indicators inputed. Shape:", df_indicators.shape, "\n")

print("Inputing crash labels data...")
df_crash = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{CRASH_LABEL}.csv", index_col=0, parse_dates=True)
df_crash = df_crash[(df_crash.index >= START_DATE) & (df_crash.index <= END_DATE)]
print("Crash labels data inputed. Shape:", df_crash.shape, "\n")

print("Inputing news data...")
df_news = pd.read_csv(f"{L1_RESULTS}/{L2_DATA}/{NEWS_CACHE}.csv", index_col=0, parse_dates=True)
df_news = df_news[(df_news.index >= START_DATE) & (df_news.index <= END_DATE)]
print("News data inputed. Shape:", df_news.shape, "\n")

Inputing price data for ^GSPC from 2024-12-20 to 2025-06-30 ...
Data inputted. Shape: (128, 6) 

Inputing indicators...
Indicators inputed. Shape: (128, 20) 

Inputing crash labels data...
Crash labels data inputed. Shape: (128, 3) 

Inputing news data...
News data inputed. Shape: (193, 1) 



In [139]:
generate_rolling_window_charts_only(
    df_ind=df_indicators,
    window_size=WINDOW_SIZE,
    save_fig_dir=FIG_DIR,
    max_windows=MAX_WINDOWS,
    ticker=TICKER
)

Generating Rolling Charts: 100%|██████████| 8/8 [00:25<00:00,  3.16s/it, Saved ^GSPC_2025-06-26 Series Plot]
